In [16]:
# 1. 라이브러리 불러오기

import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, AveragePooling2D,
    Flatten, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping
import pandas as pd

In [17]:
# 2. Fashion MNIST 데이터 불러오기
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (60000, 28, 28)
y_train shape: (60000,)
X_test shape: (10000, 28, 28)
y_test shape: (10000,)


In [ ]:
# 3. 데이터 전처리

# Fashion MNIST는 흑백 이미지이므로 채널 수는 1
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

# 픽셀 값은 0~255 범위이므로 0~1 사이로 정규화
X_train = X_train / 255.0
X_test = X_test / 255.0

print("변환 후 X_train shape:", X_train.shape)
print("변환 후 X_test shape:", X_test.shape)

변환 후 X_train shape: (60000, 28, 28, 1)
변환 후 X_test shape: (10000, 28, 28, 1)


------------

# 기본 CNN 구조

In [19]:
model1 = Sequential([
    Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same',
           input_shape=(28, 28, 1)),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'),
    MaxPooling2D(pool_size=(2, 2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

------------------

# 필터 개수를 더 많이 사용한 CNN 구조

In [20]:
model2 = Sequential([
    Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same',
           input_shape=(28, 28, 1)),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same'),
    BatchNormalization(),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(10, activation='softmax')
])

---------
# 커널 크기를 다르게 조정한 CNN 구조

In [21]:
model3 = Sequential([
    Conv2D(32, kernel_size=(5, 5), activation='relu', padding='same',
           input_shape=(28, 28, 1)),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same'),
    BatchNormalization(),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(10, activation='softmax')
])

----------------
# AveragePooling을 사용한 CNN 구조

In [31]:
model4 = Sequential([
    Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same',
           input_shape=(28, 28, 1)),
    AveragePooling2D(pool_size=(2, 2)),

    Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'),
    AveragePooling2D(pool_size=(2, 2)),

    Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same'),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

-----------

In [26]:
# 4개의 모델을 리스트에 저장
models = [
    ("Model 1: 기본 CNN", model1),
    ("Model 2: 필터 개수 증가 + BatchNormalization", model2),
    ("Model 3: 커널 크기 조정", model3),
    ("Model 4: AveragePooling 사용", model4)
]

# 결과 저장용 리스트
result_list = []

# EarlyStopping 설정
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# 각 모델을 반복해서 학습
for model_name, model in models:
    print("=" * 60)
    print(model_name)
    print("=" * 60)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train,
        y_train,
        epochs=20,
        batch_size=64,
        callbacks=[early_stop],
        validation_split=0.2,
        verbose=1
    )

    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

    print(model_name, "테스트 정확도:", test_accuracy)

    result_list.append({
        "모델": model_name,
        "테스트 손실": test_loss,
        "테스트 정확도": test_accuracy,
        "테스트 정확도(%)": round(test_accuracy * 100, 2)
    })

Model 1: 기본 CNN
Epoch 1/20
750/750 [==============================] - 8s 10ms/step - loss: 0.0503 - accuracy: 0.9808 - val_loss: 0.0392 - val_accuracy: 0.9850
Epoch 2/20
750/750 [==============================] - 7s 10ms/step - loss: 0.0489 - accuracy: 0.9820 - val_loss: 0.0424 - val_accuracy: 0.9824
Epoch 3/20
750/750 [==============================] - 7s 9ms/step - loss: 0.0419 - accuracy: 0.9838 - val_loss: 0.0388 - val_accuracy: 0.9856
Epoch 4/20
750/750 [==============================] - 7s 9ms/step - loss: 0.0436 - accuracy: 0.9831 - val_loss: 0.0404 - val_accuracy: 0.9850
Epoch 5/20
750/750 [==============================] - 7s 9ms/step - loss: 0.0398 - accuracy: 0.9853 - val_loss: 0.0475 - val_accuracy: 0.9826
Epoch 6/20
750/750 [==============================] - 7s 9ms/step - loss: 0.0395 - accuracy: 0.9854 - val_loss: 0.0597 - val_accuracy: 0.9777
Model 1: 기본 CNN 테스트 정확도: 0.920799970626831
Model 2: 필터 개수 증가 + BatchNormalization
Epoch 1/20
750/750 [============================

------------
# 추가 실험

## filler 변경

In [28]:
model5 = Sequential([
    Conv2D(16, kernel_size=(3, 3), activation='relu', padding='same',
           input_shape=(28, 28, 1)),
    AveragePooling2D(pool_size=(2, 2)),

    Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same'),
    AveragePooling2D(pool_size=(2, 2)),

    Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'),

    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

## Dropout 조정

In [29]:
model6 = Sequential([
    Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same',
           input_shape=(28, 28, 1)),
    AveragePooling2D(pool_size=(2, 2)),

    Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'),
    AveragePooling2D(pool_size=(2, 2)),

    Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same'),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(10, activation='softmax')
])

In [32]:
# 4개의 모델을 리스트에 저장
models = [
    ("Model 5: 필터 변경", model5),
    ("Model 6: Dropout 변경", model6)
]

# 결과 저장용 리스트
result_list = []

# EarlyStopping 설정
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# 각 모델을 반복해서 학습
for model_name, model in models:
    print("=" * 60)
    print(model_name)
    print("=" * 60)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train,
        y_train,
        epochs=20,
        batch_size=64,
        callbacks=[early_stop],
        validation_split=0.2,
        verbose=1
    )

    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

    print(model_name, "테스트 정확도:", test_accuracy)

    result_list.append({
        "모델": model_name,
        "테스트 손실": test_loss,
        "테스트 정확도": test_accuracy,
        "테스트 정확도(%)": round(test_accuracy * 100, 2)
    })

Model 5: 필터 변경
Epoch 1/20
750/750 [==============================] - 6s 7ms/step - loss: 0.1726 - accuracy: 0.9348 - val_loss: 0.2404 - val_accuracy: 0.9169
Epoch 2/20
750/750 [==============================] - 5s 7ms/step - loss: 0.1656 - accuracy: 0.9376 - val_loss: 0.2370 - val_accuracy: 0.9163
Epoch 3/20
750/750 [==============================] - 5s 7ms/step - loss: 0.1587 - accuracy: 0.9402 - val_loss: 0.2458 - val_accuracy: 0.9178
Epoch 4/20
750/750 [==============================] - 5s 7ms/step - loss: 0.1482 - accuracy: 0.9434 - val_loss: 0.2420 - val_accuracy: 0.9175
Epoch 5/20
750/750 [==============================] - 5s 7ms/step - loss: 0.1449 - accuracy: 0.9437 - val_loss: 0.2480 - val_accuracy: 0.9183
Model 5: 필터 변경 테스트 정확도: 0.9144999980926514
Model 6: Dropout 변경
Epoch 1/20
750/750 [==============================] - 10s 13ms/step - loss: 0.5267 - accuracy: 0.8056 - val_loss: 0.3650 - val_accuracy: 0.8658
Epoch 2/20
750/750 [==============================] - 9s 13ms/step -

-------------
# batch 조정

In [33]:
# 4개의 모델을 리스트에 저장
models = [
    ("Model 1: 기본 CNN", model1),
    ("Model 2: 필터 개수 증가 + BatchNormalization", model2),
    ("Model 3: 커널 크기 조정", model3),
    ("Model 4: AveragePooling 사용", model4),
    ("Model 5: 필터 변경", model5),
    ("Model 6: Dropout 변경", model6)
]

# 결과 저장용 리스트
result_list = []

# EarlyStopping 설정
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# 각 모델을 반복해서 학습
for model_name, model in models:
    print("=" * 60)
    print(model_name)
    print("=" * 60)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train,
        y_train,
        epochs=20,
        batch_size=64,
        callbacks=[early_stop],
        validation_split=0.2,
        verbose=1
    )

    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

    print(model_name, "테스트 정확도:", test_accuracy)

    result_list.append({
        "모델": model_name,
        "테스트 손실": test_loss,
        "테스트 정확도": test_accuracy,
        "테스트 정확도(%)": round(test_accuracy * 100, 2)
    })

Model 1: 기본 CNN
Epoch 1/20
750/750 [==============================] - 8s 11ms/step - loss: 0.0436 - accuracy: 0.9833 - val_loss: 0.0464 - val_accuracy: 0.9813
Epoch 2/20
750/750 [==============================] - 7s 9ms/step - loss: 0.0398 - accuracy: 0.9850 - val_loss: 0.0438 - val_accuracy: 0.9827
Epoch 3/20
750/750 [==============================] - 7s 9ms/step - loss: 0.0416 - accuracy: 0.9837 - val_loss: 0.0488 - val_accuracy: 0.9817
Epoch 4/20
750/750 [==============================] - 7s 10ms/step - loss: 0.0339 - accuracy: 0.9875 - val_loss: 0.0530 - val_accuracy: 0.9797
Epoch 5/20
750/750 [==============================] - 9s 12ms/step - loss: 0.0349 - accuracy: 0.9867 - val_loss: 0.0497 - val_accuracy: 0.9802
Model 1: 기본 CNN 테스트 정확도: 0.9211000204086304
Model 2: 필터 개수 증가 + BatchNormalization
Epoch 1/20
750/750 [==============================] - 14s 18ms/step - loss: 0.1612 - accuracy: 0.9422 - val_loss: 0.1567 - val_accuracy: 0.9416
Epoch 2/20
750/750 [========================